In [ ]:
"""
sgRNA niraparib log2FC analysis

This script merges CRISPResso files, collapses technical
replicates, assigns sgRNAs using a reference guide table, calculates condition
summary statistics, computes log2 fold-change relative to Day 7, and generates position plots.

Expected input files
--------------------
Tab-delimited .txt files containing at least:
    Aligned_Sequence, #Reads

Sample names should contain condition and replicate labels, for example:
    day_7_A, mock_A, 025_nira_A, 05_nira_A, 1_nira_A

Main configurable options
-------------------------
- NIRAPARIB_CONCENTRATIONS controls how many niraparib doses are included.
- CDNA_WINDOW controls the cDNA range plotted.
- MUTATION_CDNA and MUTATION_LABEL control the mutation marker.
- USE_ANNOTATIONS controls whether annotation overlays are drawn.
"""

from __future__ import annotations

import itertools
import re
from functools import reduce
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


# -----------------------------------------------------------------------------
# Input user settings & local file directories
# -----------------------------------------------------------------------------

# BASE_DIR = Path("/path/to/reversion_folder")

INPUT_DIR = BASE_DIR / "example_inputs" / "sgRNA_frequency"
OUTPUT_DIR = BASE_DIR / "example_outputs" / "sgRNA_frequency"
SGRNA_FILE = BASE_DIR / "example_inputs" / "BRCA2_sgRNAs.csv"
ANNOTATIONS_FILE = BASE_DIR / "example_inputs" / "BRCA2_annotations.csv"

# Replicate labels should match the input file names.
REPLICATES = ["A", "B", "C"]

# Baseline and untreated/control conditions.
BASELINE_CONDITION = "day_7"
MOCK_CONDITION = "mock"

# Add, remove, or reorder niraparib concentrations here.
# prefix:  column/file-name prefix
# label:   plot/CSV label
# color:   plotting color
NIRAPARIB_CONCENTRATIONS = [
    # {"prefix": "025_nira", "label": "0.25uM Nira", "color": "darkorange"},
    # {"prefix": "05_nira", "label": "0.5uM Nira", "color": "red"},
    {"prefix": "1_nira", "label": "1uM Nira", "color": "deeppink"},
]

# Plot conditions are Mock plus however many niraparib concentrations are listed above.
PLOT_CONDITIONS = [
    {"prefix": MOCK_CONDITION, "label": "Mock", "color": "deepskyblue"},
    *NIRAPARIB_CONCENTRATIONS,
]

# cDNA range to plot. Set to None to plot the full available range.
CDNA_WINDOW = (0, 200000)

# Optional guide filtering. Use None to include all sgRNAs.
# Example: GRNA_EXON_FILTER = 11
GRNA_EXON_FILTER = None

# Mutation marker.
MUTATION_CDNA = 4703
MUTATION_LABEL = "Y1569Rfs*2"
SHOW_MUTATION_MARKER = True

# Annotation overlays.
USE_ANNOTATIONS = True
SHOW_REPEATS = True
SHOW_EXON_BOUNDARIES = True

# Whole-gene/panel plot settings.
N_PANELS = 9
OUTPUT_FILENAME = "whole_gene_sgrna_log2fc.pdf"
SAVE_DPI = 300


# -----------------------------------------------------------------------------
# Figure style
# -----------------------------------------------------------------------------

FIG_WIDTH = 8
FIG_HEIGHT = 11
TITLE_FONTSIZE = 6
AXIS_LABEL_FONTSIZE = 7
XTICK_FONTSIZE = 6
YTICK_FONTSIZE = 6
LEGEND_FONTSIZE = 6

TITLE_FONTWEIGHT = "bold"
AXIS_LABEL_FONTWEIGHT = "bold"
XTICK_FONTWEIGHT = "bold"
LEGEND_FONTWEIGHT = "bold"

BAR_WIDTH = 2
GROUP_GAP = 2.0
REPLICATE_DOT_SIZE = 0.2
GRID_ALPHA = 0.3
REPEAT_ALPHA = 0.18
EXON_LINEWIDTH = 1
MUTATION_LINEWIDTH = 1
BAR_EDGEWIDTH = 0.2

YMIN = None
YMAX = None

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial"]


# -----------------------------------------------------------------------------
# Input processing
# -----------------------------------------------------------------------------

def read_processed_count_file(file_path: Path) -> pd.DataFrame | None:
    """Read one count file, clean it in memory, and label read columns by sample name."""
    try:
        df = pd.read_csv(file_path, sep="\t")
    except Exception as exc:
        print(f"Skipping {file_path.name}: could not read file ({exc})")
        return None

    drop_columns = [
        "Reference_Sequence",
        "Reference_Name",
        "Read_Status",
        "n_deleted",
        "n_inserted",
        "n_mutated",
    ]

    df = df.drop(columns=[c for c in drop_columns if c in df.columns], errors="ignore")

    if "Aligned_Sequence" not in df.columns:
        print(f"Skipping {file_path.name}: missing Aligned_Sequence column")
        return None

    df["Aligned_Sequence"] = (
        df["Aligned_Sequence"].astype(str).str.replace("-", "", regex=False)
    )

    sample_name = file_path.stem
    rename_dict = {}

    if "#Reads" in df.columns:
        rename_dict["#Reads"] = f"{sample_name}_#Reads"
    if "%Reads" in df.columns:
        rename_dict["%Reads"] = f"{sample_name}_%Reads"

    df = df.rename(columns=rename_dict)
    keep_cols = ["Aligned_Sequence"] + list(rename_dict.values())

    return df[keep_cols]


def merge_count_files(input_dir: Path) -> pd.DataFrame:
    """Merge all count files by aligned sequence."""
    dfs = [read_processed_count_file(path) for path in sorted(input_dir.glob("*.txt"))]
    dfs = [df for df in dfs if df is not None]

    if not dfs:
        raise FileNotFoundError(f"No valid .txt files found in {input_dir}")

    merged = reduce(
        lambda left, right: pd.merge(left, right, on="Aligned_Sequence", how="outer"),
        dfs,
    ).fillna(0)

    read_cols = [col for col in merged.columns if col.endswith("_#Reads")]
    merged["common_count"] = merged[read_cols].gt(0).sum(axis=1)
    merged["total_reads"] = merged[read_cols].sum(axis=1)

    return merged.sort_values(
        by=["common_count", "total_reads"],
        ascending=False,
    )


def condition_prefixes() -> list[str]:
    """Return all condition prefixes to collapse into percent-read columns."""
    return [BASELINE_CONDITION, MOCK_CONDITION] + [d["prefix"] for d in NIRAPARIB_CONCENTRATIONS]


def combine_technical_replicates(
    df: pd.DataFrame,
    prefixes: list[str] | None = None,
    replicates: list[str] = REPLICATES,
) -> pd.DataFrame:
    """Sum technical replicates and convert each sample to percent reads."""
    df = df.copy()
    prefixes = prefixes or condition_prefixes()

    drop_cols = [
        col for col in df.columns
        if "%Reads" in col or col == "total num reads" or col == "reference_sequence" or "n_" in col
    ]
    df = df.drop(columns=drop_cols, errors="ignore")

    for prefix, replicate in itertools.product(prefixes, replicates):
        sample_key = f"{prefix}_{replicate}"
        matching_cols = [col for col in df.columns if re.search(sample_key, col)]

        if not matching_cols:
            continue

        sum_col = f"{sample_key}_read_sum"
        pct_col = f"{sample_key}%reads"

        df[sum_col] = df[matching_cols].sum(axis=1)
        total_reads = df[sum_col].sum()
        df[pct_col] = (df[sum_col] / total_reads) * 100 if total_reads else 0

    read_count_cols = [col for col in df.columns if "#" in col or col.endswith("_read_sum")]
    return df.drop(columns=read_count_cols, errors="ignore")


# -----------------------------------------------------------------------------
# sgRNA annotation and summaries
# -----------------------------------------------------------------------------

def load_sgrna_table(path: Path) -> pd.DataFrame:
    """Load sgRNA metadata."""
    df = pd.read_csv(path)
    required = {"name", "sequence", "exon", "cut_position"}
    missing = required - set(df.columns)

    if missing:
        raise KeyError(f"sgRNA file is missing columns: {sorted(missing)}")

    df = df.copy()
    df["sequence"] = df["sequence"].astype(str).str.upper()
    return df


def add_sgrna_info(counts: pd.DataFrame, sgrna: pd.DataFrame) -> pd.DataFrame:
    """Assign sgRNA metadata by searching each aligned sequence for guide sequences."""
    counts = counts.copy()
    counts["Aligned_Sequence"] = counts["Aligned_Sequence"].astype(str).str.upper()

    guide_lookup = {
        row["sequence"]: (row["name"], row["sequence"], row["exon"])
        for _, row in sgrna.iterrows()
    }
    guide_pattern = re.compile("|".join(map(re.escape, guide_lookup.keys())))

    def lookup_guide(sequence: str) -> tuple[object, object, object]:
        match = guide_pattern.search(sequence)
        if match:
            return guide_lookup[match.group(0)]
        return (np.nan, np.nan, np.nan)

    counts[["gRNA_name", "gRNA_sequence", "gRNA_exon"]] = counts["Aligned_Sequence"].apply(
        lambda sequence: pd.Series(lookup_guide(sequence))
    )

    return counts


def extract_sgrna_number(name: object) -> float:
    """Extract a terminal numeric guide ID for sorting."""
    if pd.isna(name):
        return float("inf")

    match = re.search(r"(\d+)$", str(name))
    return int(match.group(1)) if match else float("inf")


def collapse_by_sgrna(df: pd.DataFrame, sgrna: pd.DataFrame) -> pd.DataFrame:
    """Collapse sequence-level rows to one row per sgRNA."""
    grouped = (
        df.groupby(["gRNA_name", "gRNA_sequence", "gRNA_exon"], dropna=False, as_index=False)
          .sum(numeric_only=True)
    )

    grouped["sgRNA_number"] = grouped["gRNA_name"].apply(extract_sgrna_number)
    grouped = grouped.sort_values(by=["gRNA_exon", "sgRNA_number"], na_position="last")
    grouped = grouped.drop(columns="sgRNA_number")

    grouped.loc[
        grouped["gRNA_name"].astype(str).str.contains("pos", case=False, na=False),
        "gRNA_exon",
    ] = "positive"
    grouped.loc[
        grouped["gRNA_name"].astype(str).str.contains("neg", case=False, na=False),
        "gRNA_exon",
    ] = "negative"

    grouped = grouped.merge(
        sgrna[["name", "cut_position"]],
        how="left",
        left_on="gRNA_name",
        right_on="name",
    ).drop(columns="name")

    return grouped


def condition_columns() -> dict[str, list[str]]:
    """Build condition-to-replicate-column mapping."""
    mapping = {
        "Day 7": [f"{BASELINE_CONDITION}_{r}%reads" for r in REPLICATES],
        "Mock": [f"{MOCK_CONDITION}_{r}%reads" for r in REPLICATES],
    }

    for dose in NIRAPARIB_CONCENTRATIONS:
        mapping[dose["label"]] = [f"{dose['prefix']}_{r}%reads" for r in REPLICATES]

    return mapping


def add_condition_stats(df: pd.DataFrame, conditions: dict[str, list[str]]) -> pd.DataFrame:
    """Add mean and standard deviation of percent reads for each condition."""
    stats = df.copy()

    for condition, columns in conditions.items():
        existing = [col for col in columns if col in df.columns]
        if not existing:
            print(f"No columns available for {condition}")
            stats[f"{condition}_mean"] = np.nan
            stats[f"{condition}_std"] = np.nan
            continue

        stats[f"{condition}_mean"] = df[existing].mean(axis=1)
        stats[f"{condition}_std"] = df[existing].std(axis=1)

    return stats


def add_log2_fold_changes(
    df: pd.DataFrame,
    comparison_prefixes: list[str] | None = None,
    reference_prefix: str = BASELINE_CONDITION,
    replicates: list[str] = REPLICATES,
) -> pd.DataFrame:
    """Calculate replicate-level and mean log2FC relative to Day 7.

    This uses the same logic as the notebook version: no pseudocount is added;
    infinite values from division by zero are converted to NaN before the mean is
    calculated.
    """
    df = df.copy()
    comparison_prefixes = comparison_prefixes or [MOCK_CONDITION] + [
        d["prefix"] for d in NIRAPARIB_CONCENTRATIONS
    ]

    for condition in comparison_prefixes:
        for replicate in replicates:
            numerator = f"{condition}_{replicate}%reads"
            denominator = f"{reference_prefix}_{replicate}%reads"
            output = f"{condition}_{replicate}_log2FC"

            if numerator not in df.columns or denominator not in df.columns:
                df[output] = np.nan
                continue

            with np.errstate(divide="ignore", invalid="ignore"):
                df[output] = np.log2(df[numerator] / df[denominator])

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    for condition in comparison_prefixes:
        replicate_cols = [f"{condition}_{replicate}_log2FC" for replicate in replicates]
        replicate_cols = [col for col in replicate_cols if col in df.columns]
        df[f"{condition}_mean_log2FC"] = df[replicate_cols].mean(axis=1, skipna=True)

    return df


def build_plot_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Keep metadata and log2FC columns for plotting."""
    metadata = ["gRNA_name", "gRNA_sequence", "gRNA_exon", "cut_position"]
    metadata = [col for col in metadata if col in df.columns]
    log2fc_cols = [col for col in df.columns if "log2fc" in col.lower()]

    return df[metadata + log2fc_cols].copy()


# -----------------------------------------------------------------------------
# Plot helpers
# -----------------------------------------------------------------------------

def load_annotations(path: Path) -> pd.DataFrame | None:
    """Load optional cDNA annotation intervals."""
    if not USE_ANNOTATIONS:
        return None

    if not path.exists():
        raise FileNotFoundError(f"Annotation file does not exist: {path}")

    annotations = pd.read_csv(path)
    required = {"Name", "Start", "end", "type"}
    missing = required - set(annotations.columns)

    if missing:
        raise KeyError(f"Annotation file is missing columns: {sorted(missing)}")

    annotations = annotations.copy()
    annotations["Start"] = pd.to_numeric(annotations["Start"], errors="coerce")
    annotations["end"] = pd.to_numeric(annotations["end"], errors="coerce")
    return annotations.dropna(subset=["Start", "end"])


def plot_condition_specs() -> list[dict[str, str]]:
    """Return plot condition metadata with log2FC column names."""
    specs = []

    for condition in PLOT_CONDITIONS:
        prefix = condition["prefix"]
        label = condition["label"]
        specs.append(
            {
                "prefix": prefix,
                "label": label,
                "color": condition["color"],
                "mean_col": f"{prefix}_mean_log2FC",
                "rep_cols": [f"{prefix}_{rep}_log2FC" for rep in REPLICATES],
            }
        )

    return specs


def prepare_position_plot_data(
    df_plot: pd.DataFrame,
    cdna_window: tuple[int, int] | None = CDNA_WINDOW,
    exon_filter: int | str | None = GRNA_EXON_FILTER,
) -> pd.DataFrame:
    """Clean and optionally filter sgRNA log2FC data for plotting."""
    df = df_plot.copy()
    df["cut_position"] = pd.to_numeric(df["cut_position"], errors="coerce")
    df = df.dropna(subset=["cut_position"])

    if exon_filter is not None and "gRNA_exon" in df.columns:
        df = df[df["gRNA_exon"].astype(str) == str(exon_filter)]

    if cdna_window is not None:
        start, end = cdna_window
        df = df[(df["cut_position"] >= start) & (df["cut_position"] <= end)]

    df = df.set_index("cut_position").sort_index()

    specs = plot_condition_specs()
    mean_cols = [spec["mean_col"] for spec in specs if spec["mean_col"] in df.columns]

    if mean_cols:
        df = df.loc[~df[mean_cols].isna().all(axis=1)]

    return df


def cdna_to_panel_x(cdna_coord: float, cut_positions: np.ndarray, x_slots: np.ndarray) -> float:
    """Map a cDNA coordinate onto the synthetic x-axis used for one panel."""
    cdna_coord = float(cdna_coord)
    cdna_coord = max(cut_positions.min(), min(cut_positions.max(), cdna_coord))
    return float(np.interp(cdna_coord, cut_positions, x_slots))


def add_annotation_overlays(
    ax: plt.Axes,
    annotations: pd.DataFrame | None,
    left_cdna: float,
    right_cdna: float,
    cut_positions: np.ndarray,
    x_slots: np.ndarray,
) -> None:
    """Draw repeat regions and exon boundaries, if requested."""
    if annotations is None or not USE_ANNOTATIONS:
        return

    if SHOW_REPEATS:
        repeats = annotations.loc[annotations["type"] == "repeat"].sort_values("Start")
        for _, repeat in repeats.iterrows():
            start, end = float(repeat["Start"]), float(repeat["end"])
            if end < left_cdna or start > right_cdna:
                continue

            x0 = cdna_to_panel_x(max(start, left_cdna), cut_positions, x_slots)
            x1 = cdna_to_panel_x(min(end, right_cdna), cut_positions, x_slots)
            ax.axvspan(x0, x1, alpha=REPEAT_ALPHA, zorder=0)

    if SHOW_EXON_BOUNDARIES:
        exons = annotations.loc[annotations["type"] == "exon"].sort_values("Start")
        boundaries = exons["Start"].astype(float).tolist()
        if len(exons):
            boundaries.append(float(exons["end"].iloc[-1]))

        for boundary in boundaries:
            if boundary < left_cdna or boundary > right_cdna:
                continue

            xb = cdna_to_panel_x(boundary, cut_positions, x_slots)
            ax.axvline(
                xb,
                color="black",
                linestyle="--",
                linewidth=EXON_LINEWIDTH,
                alpha=0.8,
                zorder=0.5,
            )


def plot_sgrna_log2fc_by_position(
    df_plot: pd.DataFrame,
    output_path: Path,
    annotations: pd.DataFrame | None = None,
    n_panels: int = N_PANELS,
    cdna_window: tuple[int, int] | None = CDNA_WINDOW,
    exon_filter: int | str | None = GRNA_EXON_FILTER,
) -> None:
    """Plot sgRNA log2FC values by cDNA cut position."""
    dfp = prepare_position_plot_data(df_plot, cdna_window=cdna_window, exon_filter=exon_filter)

    if dfp.empty:
        raise ValueError("No sgRNAs remain after filtering. Check CDNA_WINDOW and GRNA_EXON_FILTER.")

    specs = plot_condition_specs()
    group_width = len(specs) * BAR_WIDTH + GROUP_GAP
    chunks = np.array_split(np.arange(len(dfp)), n_panels)

    fig, axes = plt.subplots(
        n_panels,
        1,
        figsize=(FIG_WIDTH, FIG_HEIGHT),
        sharey=True,
        constrained_layout=True,
    )

    if n_panels == 1:
        axes = [axes]

    y_values = []

    for panel_index, ax in enumerate(axes):
        idx = chunks[panel_index]
        dfw = dfp.iloc[idx].copy()

        if dfw.empty:
            ax.set_axis_off()
            continue

        left_cdna = float(dfw.index.min())
        right_cdna = float(dfw.index.max())
        cut_positions = dfw.index.to_numpy(dtype=float)
        x_slots = np.arange(len(dfw)) * group_width

        add_annotation_overlays(
            ax,
            annotations,
            left_cdna,
            right_cdna,
            cut_positions,
            x_slots,
        )

        for i, spec in enumerate(specs):
            mean_col = spec["mean_col"]
            if mean_col not in dfw.columns:
                continue

            mean_values = dfw[mean_col].to_numpy(dtype=float)
            finite = np.isfinite(mean_values)
            if not finite.any():
                continue

            xs = x_slots[finite] + (i - (len(specs) - 1) / 2) * BAR_WIDTH
            hs = mean_values[finite]
            y_values.extend(hs[np.isfinite(hs)])

            ax.bar(
                xs,
                hs,
                width=BAR_WIDTH,
                label=spec["label"] if panel_index == 0 else None,
                color=spec["color"],
                edgecolor="black",
                linewidth=BAR_EDGEWIDTH,
                zorder=1,
            )

            rep_cols = [col for col in spec["rep_cols"] if col in dfw.columns]
            if rep_cols:
                reps = dfw.loc[:, rep_cols].to_numpy(dtype=float)[finite, :]
                for rep_index in range(reps.shape[1]):
                    rep_values = reps[:, rep_index]
                    y_values.extend(rep_values[np.isfinite(rep_values)])
                    ax.scatter(
                        xs,
                        rep_values,
                        color="black",
                        s=REPLICATE_DOT_SIZE,
                        zorder=3,
                    )

        if SHOW_MUTATION_MARKER and left_cdna <= MUTATION_CDNA <= right_cdna:
            mutation_x = cdna_to_panel_x(MUTATION_CDNA, cut_positions, x_slots)
            ax.axvline(
                mutation_x,
                color="red",
                linestyle="--",
                linewidth=MUTATION_LINEWIDTH,
                zorder=10,
            )

        pad = group_width / 2
        ax.set_xlim(x_slots[0] - pad, x_slots[-1] + pad)
        ax.margins(x=0)
        ax.grid(axis="y", linestyle="-", alpha=GRID_ALPHA)
        ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)

        ax.set_xticks(x_slots)
        ax.set_xticklabels(
            cut_positions.round().astype(int).astype(str),
            rotation=90,
            fontsize=XTICK_FONTSIZE,
            fontweight=XTICK_FONTWEIGHT,
        )

        ax.set_ylabel(
            r"$\mathrm{Log}_{2}(\mathrm{Treatment}/\mathrm{Day\ 7})$",
            fontsize=AXIS_LABEL_FONTSIZE,
            fontweight=AXIS_LABEL_FONTWEIGHT,
        )

        ax.set_title(
            f"cDNA {int(left_cdna)}–{int(right_cdna)}",
            fontsize=TITLE_FONTSIZE,
            fontweight=TITLE_FONTWEIGHT,
            pad=6,
        )

    if YMIN is not None or YMAX is not None:
        for ax in axes:
            current_ymin, current_ymax = ax.get_ylim()
            ax.set_ylim(
                YMIN if YMIN is not None else current_ymin,
                YMAX if YMAX is not None else current_ymax,
            )

    axes[-1].set_xlabel(
        "sgRNA cut position (cDNA)",
        fontsize=AXIS_LABEL_FONTSIZE,
        fontweight=AXIS_LABEL_FONTWEIGHT,
    )

    handles, labels = axes[0].get_legend_handles_labels()

    if USE_ANNOTATIONS and SHOW_REPEATS:
        handles.append(Patch(alpha=REPEAT_ALPHA, label="BRC repeats"))
        labels.append("BRC repeats")

    if USE_ANNOTATIONS and SHOW_EXON_BOUNDARIES:
        handles.append(
            Line2D(
                [0],
                [0],
                color="black",
                linestyle="--",
                linewidth=EXON_LINEWIDTH,
                label="Exon boundary",
            )
        )
        labels.append("Exon boundary")

    if SHOW_MUTATION_MARKER:
        handles.append(
            Line2D(
                [0],
                [0],
                color="red",
                linestyle="--",
                linewidth=MUTATION_LINEWIDTH,
                label=MUTATION_LABEL,
            )
        )
        labels.append(MUTATION_LABEL)

    axes[0].legend(
        handles=handles,
        labels=labels,
        fontsize=LEGEND_FONTSIZE,
        bbox_to_anchor=(1.01, 1),
        loc="upper left",
        prop={"weight": LEGEND_FONTWEIGHT, "size": LEGEND_FONTSIZE},
        frameon=False,
    )

    fig.savefig(output_path, format="pdf", dpi=SAVE_DPI, bbox_inches="tight")
    plt.close(fig)


# -----------------------------------------------------------------------------
# Main analysis
# -----------------------------------------------------------------------------

def main() -> None:
    if not INPUT_DIR.exists():
        raise FileNotFoundError(f"Input directory does not exist: {INPUT_DIR}")
    if not SGRNA_FILE.exists():
        raise FileNotFoundError(f"sgRNA file does not exist: {SGRNA_FILE}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    sgrna = load_sgrna_table(SGRNA_FILE)
    annotations = load_annotations(ANNOTATIONS_FILE)

    merged = merge_count_files(INPUT_DIR)
    merged.to_csv(OUTPUT_DIR / "merged_output.csv", index=False)

    replicate_counts = combine_technical_replicates(merged)
    with_guides = add_sgrna_info(replicate_counts, sgrna)
    sgrna_summary = collapse_by_sgrna(with_guides, sgrna)
    sgrna_summary.to_csv(OUTPUT_DIR / "merged_output_with_gRNA_info.csv", index=False)

    stats = add_condition_stats(sgrna_summary, condition_columns())
    stats.to_csv(OUTPUT_DIR / "gRNA_percentages.csv", index=False)

    log2fc = add_log2_fold_changes(stats)
    log2fc.to_csv(OUTPUT_DIR / "gRNA_log2FC_full.csv", index=False)

    df_plot = build_plot_dataframe(log2fc)
    df_plot.to_csv(OUTPUT_DIR / "gRNA_log2FC_plot_input.csv", index=False)

    plot_sgrna_log2fc_by_position(
        df_plot,
        OUTPUT_DIR / OUTPUT_FILENAME,
        annotations=annotations,
        n_panels=N_PANELS,
        cdna_window=CDNA_WINDOW,
        exon_filter=GRNA_EXON_FILTER,
    )

    print(f"Analysis complete. Results saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()